In [2]:
import textgrid
import math
import cv2
import numpy as np
import os
from tqdm import tqdm

In [3]:
# ########## MAIN MAPPING

# phoneme2viseme = {
#     'AA':0,
#     'AE':1,
#     'AH':3,
#     'AO':5,
#     'AW':6,
#     'AY':2,
#     'AX':0,
#     'B':18,
#     'CH':16,
#     'D':16,
#     'DH':16,
#     'EH':3,
#     'ER':3,
#     'EY':1,
#     'F':19,
#     'G':3,
#     'HH':3,
#     'IH':8,
#     'IY':7,
#     'JH':16,
#     'K':15,
#     'L':17,
#     'M':18,
#     'N':17,
#     'NG':7,
#     'OW':10,
#     'OY':10,
#     'P':18,
#     'R':17,
#     'S':15,
#     'SH':15,
#     'T':16,
#     'TH':16,
#     'UH':13,
#     'UW':13,
#     'UX':13,
#     'V':19,
#     'W':13,
#     'Y':7,
#     'Z':16,
#     'ZH':16
# }

In [4]:
phoneme2viseme = {
    'AA':2,
    'AE':7, 
    'AH':16,
    'AO':8,
    'AW':8,
    'AY':4,
    'AX':2,
    'B':1, 
    'CH':14,
    'D':13,
    'DH':14,
    'EH':7,
    'ER':7,
    'EY':2,
    'F':15,
    'G':11,
    'HH':11,
    'IH':1,
    'IY':4,
    'JH':14,
    'K':11,
    'L':13,
    'M':1,
    'N':13,
    'NG':11,
    'OW':8,
    'OY':8,
    'P':1,
    'R':13,
    'S':14,
    'SH':10,
    'T':14,
    'TH':14,
    'UH':17,
    'UW':17,
    'UX':17,
    'V':15,
    'W':17,
    'Y':4,
    'Z':12,
    'ZH':12
}

In [11]:
# set paths
ALIGNER_ROOT = '/mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/ForcedAligner'
ASSETS_ROOT = '/mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/PresetGeneration/OUTPUT/DRAWINGS'
FFMPEG_PATH = '/mnt/users_scratch/astitva/WORKSPACE/ffmpeg/installation/bin'
VIDEO_SAVE_DIR = '/mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/VIDEO_ANIM'

### MOUTH + EYES (ANIMATION)

In [7]:
def composite(base, mouth, eyes, use_default_mouth=False, use_default_eyes=False):
    mask_mouth = mouth[:,:,3]/255
    mask_eyes = eyes[:,:,3]/255
    mask = mask_mouth + mask_eyes
    if use_default_eyes:
      mask = mask_mouth
    if use_default_mouth:
        mask = mask_eyes
    mask_im = np.repeat(mask[..., np.newaxis], 3, axis=2)
    mask_mouth_im = np.repeat(mask_mouth[..., np.newaxis], 3, axis=2)
    mask_eyes_im = np.repeat(mask_eyes[..., np.newaxis], 3, axis=2)
    composited = base*(1-mask_im) 
    if not use_default_eyes:
        composited += mask_eyes_im*eyes[:,:,:3]
    if not use_default_mouth:
        composited += mask_mouth_im*mouth[:,:,:3]
    return composited.astype('uint8')

In [8]:
filename = 'babyshark'

tg_path = f'{ALIGNER_ROOT}/outputs/{filename}.TextGrid'
tg = textgrid.TextGrid.fromFile(tg_path)

words = tg[0]
phonemes = tg[1]

words_phonemes=[]
phoneme_idx = 0
for w in words:
    current_set = []
    time = w.duration()
    start = 0
    while(time!=start):
        ph = phonemes[phoneme_idx]
        start += ph.duration()
        current_set.append(ph)
        phoneme_idx += 1
    words_phonemes.append(current_set)

words

IntervalTier(words, [Interval(0.0, 0.88, None), Interval(0.88, 1.11, baby), Interval(1.11, 1.91, None), Interval(1.91, 2.04, shark), Interval(2.04, 2.41, doo), Interval(2.41, 2.58, None), Interval(2.58, 2.76, doo), Interval(2.76, 3.04, doo), Interval(3.04, 3.51, None), Interval(3.51, 4.01, doo), Interval(4.01, 4.16, None), Interval(4.16, 4.28, doo), Interval(4.28, 4.45, doo), Interval(4.45, 4.55, None), Interval(4.55, 4.99, baby), Interval(4.99, 5.09, None), Interval(5.09, 5.35, shark), Interval(5.35, 5.46, doo), Interval(5.46, 5.57, None), Interval(5.57, 5.81, doo), Interval(5.81, 5.85, None), Interval(5.85, 5.96, doo), Interval(5.96, 6.1, doo), Interval(6.1, 6.25, None), Interval(6.25, 6.36, doo), Interval(6.36, 6.54, doo), Interval(6.54, 6.64, None), Interval(6.64, 7.09, baby), Interval(7.09, 7.18, None), Interval(7.18, 7.43, shark), Interval(7.43, 7.53, doo), Interval(7.53, 7.66, None), Interval(7.66, 7.92, doo), Interval(7.92, 7.96, None), Interval(7.96, 8.07, doo), Interval(8.07,

In [9]:
# eye_word_mapping = {'please':2, 'thankyou':3}
eye_word_mapping = {'mommy':3,'daddy':0,'grandma':6,'grandpa':7, 'hunt':5}


In [13]:
characters = ['07aedcb335a04981a016c0c7efed77ba']
# characters = ['07b97debed234daaa04313b000637b81', '07ad6ccc1ac34288ab7c4f0a013ba3c3','07aedcb335a04981a016c0c7efed77ba', '07b1a55d68b9425caccb1aadcc58379a', '07b6c37d7a6944ee98f544d633defeeb', '07b8bf4a421744c9b7f985cf6e8fe544', '07c4c1c55b5b4098bf8f5b96defd6d2c']
# characters = ['0a3b9f4c787743458c7ca1cc77b902ea', '0a4a8a95ac934f4e8d7b58561f7913c9', '0a0be5b3db37407cb434c5e0dc3cf70b', '0a0cd1cc72f44d418e4884c7a402b030', '0a6bf1b9d15842b6822a92a6b536faf1', '0a5b805185614f839b5f015b650970dc']

crop_face = True
fps=240

suffix = ''
if crop_face:
    suffix = '_face'

skip_start_frames = 0
if filename=='babyshark':
    skip_start_frames = 1063

for character_id in characters:

    MOUTH_ROOT = f'{ASSETS_ROOT}/mouth/{character_id}'
    EYES_ROOT = f'{ASSETS_ROOT}/eyes/{character_id}'
    SAVE_ROOT = f'{VIDEO_SAVE_DIR}/{character_id}/'
    os.makedirs(SAVE_ROOT, exist_ok=True)

    asset_dict = {}
    asset_dict['inpainted_mouth_only'] = cv2.imread(f'{MOUTH_ROOT}/metadata/inpainted_mouth_only.png')
    asset_dict['inpainted_eyes_only'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_eyes_only.png')
    asset_dict['inpainted_eyes_mouth'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_eyes_mouth.png')
    asset_dict['inpainted_face_mouth_only'] = cv2.imread(f'{MOUTH_ROOT}/metadata/inpainted_face_mouth_only.png')
    asset_dict['inpainted_face_eyes_only'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_face_eyes_only.png')
    asset_dict['inpainted_face_eyes_mouth'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_face_eyes_mouth.png')

    mouth_files = os.listdir(f'{MOUTH_ROOT}/assets/')
    for m in mouth_files:
        asset_dict[m[:-4]] = cv2.imread(f'{MOUTH_ROOT}/assets/{m}', -1)

    eye_files = os.listdir(f'{EYES_ROOT}/assets/')
    for e in eye_files:
        asset_dict[e[:-4]] = cv2.imread(f'{EYES_ROOT}/assets/{e}', -1)

    asset_dict.keys()
    
    base = asset_dict[f'inpainted{suffix}_mouth_only']
    eyes_id = 0
    blink_id = 2
    blink_gap = 2 #seconds
    num_blink_frames = 12
    
    USE_DEFAULT_MOUTH = False
    USE_DEFAULT_EYES = True
    
    video=cv2.VideoWriter(f'{SAVE_ROOT}/{filename}_no_audio.mp4',cv2.VideoWriter_fourcc(*'DIVX'),fps,(1024,1024))
    buffer = np.ones((1024,1024,3)).astype('uint8')*255
    current_frame_count = 0
    
    for i in tqdm(range(len(words))):
        total_duration = words[i].duration()
        word_frame_count = int(fps*total_duration)
        cumulative_frame_count = 0
    
        #eyes asset
        try:
            eyes_id = eye_word_mapping[words[i].mark]
            # base = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
            base = asset_dict[f'inpainted{suffix}_eyes_mouth']
            USE_DEFAULT_EYES=False
        except:
            pass
    
        # eyes = cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{eyes_id}_asset.png', -1)
        eyes = asset_dict[f'eyes_{eyes_id}{suffix}']

        
        for p in words_phonemes[i]:
            frame_count = math.ceil(fps*p.duration())
    
            #default mouth asset
            mouth_id = 0
            # buffer_mouth = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
            buffer_mouth = asset_dict[f'mouth_{mouth_id}{suffix}']
            
            # default
            buffer = composite(base, buffer_mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
            
            viseme_id=''
            if p.mark!='':
                try:
                    mouth_id = phoneme2viseme[p.mark[:2]]
                    # mouth = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
                    mouth = asset_dict[f'mouth_{mouth_id}{suffix}']
                    buffer_mouth = mouth
                    buffer = composite(base, mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
                except:
                    pass
            # print(USE_DEFAULT_EYES)
            for _ in range(frame_count):
                if cumulative_frame_count>=word_frame_count:
                    break
                #blink
                if current_frame_count%(fps*blink_gap)<num_blink_frames:
                    # blink_eyes =  cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{blink_id}_asset.png', -1)
                    blink_eyes = asset_dict[f'eyes_{blink_id}{suffix}']
                    # blink_base = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
                    blink_base = asset_dict[f'inpainted{suffix}_eyes_mouth']

                    if current_frame_count>skip_start_frames:
                        video.write(composite(blink_base, buffer_mouth, blink_eyes))
                else:
                    if current_frame_count>skip_start_frames:
                        video.write(buffer)
                cumulative_frame_count += 1
                current_frame_count += 1
                
    video.release()

    command = f'{FFMPEG_PATH}/ffmpeg -i {SAVE_ROOT}/{filename}_no_audio.mp4 -i {ALIGNER_ROOT}/inputs/{filename}/{filename}_trimmed.wav -map 0:v:0 -map 1:a:0 -c:v copy -framerate {60}/1 {SAVE_ROOT}/{filename}.mp4'
    os.system(command)

    print(f'Video saved --> {SAVE_ROOT}/{filename}.mp4')

OpenCV: FFMPEG: tag 0x58564944/'DIVX' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 209/209 [05:27<00:00,  1.57s/it]
ffmpeg version 5.0 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 11 (GCC)
  configuration: --prefix=/mnt/users_scratch/astitva/WORKSPACE/ffmpeg/installation --disable-debug --disable-x86asm
  libavutil      57. 17.100 / 57. 17.100
  libavcodec     59. 18.100 / 59. 18.100
  libavformat    59. 16.100 / 59. 16.100
  libavdevice    59.  4.100 / 59.  4.100
  libavfilter     8. 24.100 /  8. 24.100
  libswscale      6.  4.100 /  6.  4.100
  libswresample   4.  3.100 /  4.  3.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/VIDEO_ANIM/07aedcb335a04981a016c0c7efed77ba//babyshark_no_audio.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2m

Video saved --> /mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/VIDEO_ANIM/07aedcb335a04981a016c0c7efed77ba//babyshark.mp4


frame=11238 fps=7329 q=-1.0 Lsize=   34332kB time=00:00:46.84 bitrate=6003.4kbits/s speed=30.6x    
video:33497kB audio:755kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.236198%
[aac @ 0x2974180] Qavg: 603.108
